In [1]:
import os, requests, asyncio
from IPython.display import display, Markdown
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool, SQLiteSession
from pydantic import BaseModel
from typing_extensions import TypedDict
from datetime import datetime
from agents import RunResult  # Make sure RunResult is imported
# from helper import format_report_to_markdown
from agents import handoff, RunContextWrapper, CodeInterpreterTool, input_guardrail, GuardrailFunctionOutput, TResponseInputItem
from agents.extensions import handoff_filters
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

In [2]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

print("✅ API keys loaded")
print(f"OpenAI: {openai_api_key[:5]}***")
print(f"Tavily: {tavily_api_key[:5]}***")

✅ API keys loaded
OpenAI: sk-pr***
Tavily: tvly-***


In [3]:
def print_markdown(txt: str):
    display(Markdown(txt))

In [4]:
main_model = "gpt-4.1-mini"
small_model = "gpt-4.1-nano"

In [5]:
# Define the Tavily search function
class TavilyParams(TypedDict):
    query: str
    max_results: int


@function_tool
def tavily_search(params: TavilyParams) -> str:
    """Return a newline‑joined summary of Tavily results."""
    url = "https://api.tavily.com/search"
    payload = {
        "api_key": tavily_api_key,
        "query": params["query"],
        "max_results": params.get("max_results", 3),
    }
    resp = requests.post(url, json = payload, headers = {"Content-Type": "application/json"})
    if resp.status_code != 200:
        return f"Tavily error {resp.status_code}"
    items = resp.json().get("results", [])
    return "\n".join([f"- {itm['title']}: {itm['content']}" for itm in items]) or "No hits"

In [6]:
# We will define a Planner Agent that breaks down user research requests into 2-3 targeted web searches with reasons and queries

# This section defines a guardrail to prevent the agent from engaging in political topics. 
# It uses a dedicated LLM agent to classify the input as political or not, and triggers a tripwire if political content is detected.

# Define the output schema for the political topic guardrail.
# Whenever we classify something as political, we’ll store two fields:
# (1) is_political: a boolean flag (True/False).
# (2) reasoning: text explaining why.
# This makes the guardrail’s output predictable and structured.

class PoliticalTopicOutput(BaseModel):
    is_political: bool
    reasoning: str


# This builds an Agent (from the OpenAI Agents SDK) to detect political content.
# The instructions tell the model exactly what to do.
# The output_type enforces that the agent must return a PoliticalTopicOutput.

politics_guardrail_agent = Agent(name = "Guardrail check",
                                 instructions = "Check if the user is asking about political topics, politicians, elections, government policy, or anything related to politics. If so, set is_political to true and explain why in reasoning.",
    output_type = PoliticalTopicOutput)


# @input_guardrail: decorator that marks the function 
# The function takes:
# ctx: a wrapper object holding context/state for the current run (things like conversation history, metadata, or session info).
# input: what the user typed in (either a string, or a structured list of TResponseInputItem).
# The function returns a GuardrailFunctionOutput: what did we find, and do we trip the guardrail?


@input_guardrail
async def politics_guardrail(ctx: RunContextWrapper[None], agent: Agent, input: str | list[TResponseInputItem]) -> GuardrailFunctionOutput:
    
    # Use the guardrail agent to classify the input.
    result = await Runner.run(politics_guardrail_agent, input, context = ctx.context)

    # output_info: stores the full structured output (is_political + reasoning).
    # tripwire_triggered: this is the key flag. If is_political == True, then the guardrail has been triggered.
    return GuardrailFunctionOutput(output_info = result.final_output,
                                   tripwire_triggered = result.final_output.is_political)

In [7]:
# Define a data model for one search plan item where each item contains:
# - reason: why this particular search is being done
# - query: the actual search string to run

class SearchPlanItem(BaseModel):
    reason: str
    query: str

# Define a data model for the complete search plan
# This contains a list of SearchPlanItems

class SearchPlan(BaseModel):
    searches: list[SearchPlanItem]

# Get today's date in YYYY-MM-DD format
date = datetime.now().strftime("%Y-%m-%d")

# Create an AI agent called "Planner"
planner_agent = Agent(name = "Planner",
                      instructions = f"""Current date: {date} \n Context: You are a research planner agent tasked with designing a comprehensive research plan for a user request. 
        You have access to web search tools and should utilize the current date ({date}) when planning. 
        Instruction: Break down the user's request into 3 distinct web searches, each with a clear reason and a specific query. 
        Ensure coverage of recent news, company fundamentals, risks, sentiment, and broader context. 
        Input: The user's research request and the current date. 
        Output: A list of search plan items, each with a 'reason' and a 'query', formatted as a JSON object matching the SearchPlan schema.""",
    model = main_model,
    output_type = SearchPlan,
    input_guardrails = [politics_guardrail]) # THIS IS THE GUARDRAIL THAT PREVENTS POLITICAL TOPIC ASKS

In [8]:
# Let's test the AI Agent with the Guardrail!
from agents import Runner
# q1 = "Why is Trump meeting with Putin this week?"
q1 = "solid state battery companies"
run1 = await Runner.run(starting_agent = planner_agent, input = q1)
print_markdown(f"### 🤖 Agent’s Answer\n{run1.final_output}")

### 🤖 Agent’s Answer
searches=[SearchPlanItem(reason='To find the most recent news and developments about solid state battery companies as of 2026.', query='latest news on solid state battery companies 2026'), SearchPlanItem(reason='To understand the key players, their fundamentals, and technological capabilities in the solid state battery industry.', query='top solid state battery companies company profiles and technology overview 2026'), SearchPlanItem(reason='To assess risks, market sentiment, and investment potential related to solid state battery companies in 2026.', query='solid state battery companies investment risks market sentiment 2026')]

In [9]:
# Let's define the Search Agent, which finds and summarizes the most recent, relevant information for a research query
class Summary(BaseModel):
    summary: str

search_agent = Agent(name = "Searcher",
                     instructions = """Context: You are a search specialist agent with access to the Tavily web search tool. 
        Your goal is to provide up-to-date, relevant information for a research task. 
        Instruction: Use Tavily search to find the most recent and pertinent information related to the user's query. 
        Summarize your findings clearly and concisely in no more than 200 words. 
        Input: The user's search query. 
        Output: A concise summary (≤200 words) of the most relevant and recent information found via Tavily search.""",
    tools = [tavily_search],
    model = main_model,
    output_type = Summary)

In [10]:
# Let's define the fundamentals Analysis Agent

fundamentals_agent = Agent(
    name = "FundamentalsAnalyst",
    instructions = """
    Context: You are a financial analyst specializing in company fundamentals.
    Instruction: Carefully analyze the provided notes to assess the company's financial fundamentals, including revenue, growth, and margins.
    Input: Notes containing relevant financial data and qualitative information about the company.
    Output: A concise summary (≤200 words) highlighting key points about the company's revenue, growth trajectory, and profit margins.
    Tools: The following tools are available for comprehensive research on the company:
    - tavily_search: Search the web for information about the company.
    """,
    output_type = Summary,
    model = main_model,
    tools = [tavily_search])

In [11]:
# Let's define a writer Agent which synthesizes Search and Analyst Insights into a Cohesive Investment Report

# Define the output format for the writer agent using Pydantic's BaseModel
# This ensures the agent always returns:
# - short_summary: a brief 2–3 sentence executive summary
# - markdown_report: a detailed markdown-formatted report
# - follow_up_questions: 3–5 suggested research questions

class FinalReport(BaseModel):
    short_summary: str
    markdown_report: str
    follow_up_questions: list[str]


# Define an asynchronous helper function to extract the "summary" field
# from the final_output of another agent’s run results.
# This is used so the writer agent can easily consume outputs of other tools.

async def extract_summary(run_result: RunResult) -> str:
    """Extracts the 'summary' field from the final_output of an agent run."""
    return run_result.final_output.summary

# Define the writer agent
writer_agent = Agent(name = "Writer",
                     instructions = ("""Context: You are an expert research writer preparing a comprehensive investment report on a company, using diverse information sources. Your audience is sophisticated and expects clarity, depth, and actionable insights.
        Instruction: Thoroughly analyze the provided search snippets and analyst summaries. Synthesize these into a cohesive, well-structured markdown report of at least 600 words. Your report must: (1) begin with a concise 2-3 sentence executive summary capturing the most critical findings; (2) integrate and cross-reference key facts, trends, and perspectives from all sources, highlighting both consensus and disagreement; (3) organize content with clear headings and logical flow; (4) maintain objectivity, cite evidence, and avoid speculation; (5) conclude with 3-5 insightful, specific follow-up research questions that would meaningfully advance understanding or address unresolved issues. 
        Ensure the writing is precise, professional, and tailored for an investment decision-making context.
        You must always use the 'search' tool to gather and incorporate up-to-date information in your report. The other tools—'fundamentals'—are optional and should only be used if the user specifically requests those analyses or if you determine that including them would significantly enhance the report's quality or relevance.

        Input: A set of search snippets and analyst summaries containing relevant information about the company, and any user instructions specifying which analyses to include.
        Output: A markdown-formatted report (minimum 600 words) including an executive summary and 3-5 well-crafted follow-up research questions.
        Tools: The following tools are available for comprehensive research on the company:
        - fundamentals: Get fundamentals analysis (optional)
        - search: Get search results (required)
        """
    ),
    model = main_model,
    output_type = FinalReport,

    # Note that fundamentals_agent and search_agent are other AI agents we defined earlier.
    # The method .as_tool(...) wraps each of them into a callable tool so that the writer_agent can use them.
    # This means the writer agent is not working alone — it can “call” the search agent (required) and fundamentals agent (optional) as helper sub-agents.
        
    tools = [
        fundamentals_agent.as_tool(
            "fundamentals",
            "Get fundamentals analysis",
            custom_output_extractor = extract_summary,  # Use the async function here
        ),
        search_agent.as_tool(
            "search",
            "Get search results",
            custom_output_extractor = extract_summary,  # Use the async function here
        ),
    ],

)



In [12]:
# Let's test the AI Agent
q1 = "Do a deep dive on the latest news in Tesla stock. I also need the fundamental analysis of the company"
run1 = await Runner.run(starting_agent = writer_agent, input = q1)
print_markdown(f"### 🤖 Agent’s Answer\n{run1.final_output}")

### 🤖 Agent’s Answer
short_summary="Tesla's stock has shown mixed recent performance with significant volatility, reflecting investor concerns following underwhelming Q1 delivery results and varied analyst ratings. Fundamentally, the company faces challenges including slower revenue growth and margin compression but is strategically pivoting towards higher-margin software and autonomous services that could enhance profitability over the next few years." markdown_report="# Tesla Stock and Fundamentals Deep Dive\n\n## Executive Summary\nTesla Inc. (TSLA) recently exhibited significant stock price volatility amid mixed quarterly performance and investor sentiment. While the stock has declined over the short term, it retains strong year-over-year gains. Fundamental analysis reveals a company grappling with slowing revenue growth and margin pressure, yet pursuing strategic shifts towards recurring software-based revenue streams that could bolster long-term profitability.\n\n## Recent Stock Performance and Market Sentiment\nAs of early April 2026, Tesla's stock has experienced a series of fluctuations. The share price on April 6 ranged between $337.47 and $351.67, closing at $361.26, representing a 7.0% rise from the low of the day. However, over the past week, Tesla's stock dropped by approximately 4.11%, and over the last month, it declined nearly 13%. Despite these short-term dips, Tesla maintains a robust approximately 55% increase year-over-year, showcasing underlying long-term investor confidence.\n\nThis volatility can be attributed in part to investor reactions to Tesla’s Q1 delivery results, which fell short of market expectations, prompting a selloff and variations in analyst ratings. After-hours trading data shortly after reflected a moderate recovery with a 2.93% gain, indicating mixed but cautious optimism among investors.\n\n## Fundamental Analysis Overview\n### Revenue Growth\nTesla's revenue growth has decelerated in recent years, with a current projection of around 16.9% annual growth. This marks a slowdown compared to earlier years when Tesla experienced rapid revenue expansion driven primarily by vehicle deliveries.\n\n### Profitability and Margins\nA significant concern for investors is Tesla's shrinking gross margins. The automotive gross margin has fallen from approximately 25.6% three years ago to about 18% more recently. This compression stems from multiple factors including intensifying cost pressures, competitive pricing strategies, and increased costs associated with supply chain and materials.\n\nTesla's overall profit margins currently stand near 6.3%, with company targets aiming to increase these margins to approximately 10.4% within the next three years. Achieving this will depend heavily on Tesla’s ability to monetize higher-margin recurring revenue streams such as Full Self-Driving (FSD) subscriptions and autonomous driving services.\n\n### Strategic Shifts and Revenue Mix\nTesla's shift from predominantly vehicle sales towards software and autonomous service subscriptions represents a pivotal strategic move. These high-margin services could stabilize and potentially improve overall profitability, offsetting margin challenges in the automotive segment. The company’s push for Full Self-Driving subscriptions reflects an effort to create recurring revenues less sensitive to vehicle sales fluctuations.\n\n### Financial Strength and Risks\nTesla maintains strong cash flow generation capabilities and a sizable asset base, providing financial flexibility to invest in innovation and expansion. However, investors should remain vigilant about risks such as margin compression from pricing pressures, potential delays in autonomous technology deployment, and ongoing valuation challenges given Tesla’s growth expectations.\n\n## Conclusion\nTesla’s stock performance in the short term is marked by volatility driven by delivery shortfalls and cautious market sentiment, although the long-term growth narrative remains largely intact. The company faces near-term challenges around revenue growth and profitability, yet its strategic focus on software and autonomous services presents a promising avenue for margin improvement. Investors weighing Tesla stock should consider both the risks posed by margin pressures and the potential for enhanced recurring revenues as key factors influencing future returns.\n\n## Follow-up Research Questions\n1. How sustainable is Tesla’s revenue growth in the face of increasing competition in both electric vehicles and autonomous software?\n2. What are the critical milestones and regulatory hurdles impacting the scaling and monetization of Tesla's Full Self-Driving subscriptions?\n3. How will Tesla manage cost pressures and supply chain challenges to stabilize or improve automotive gross margins?\n4. What impact could macroeconomic factors such as interest rate changes and inflation have on Tesla’s valuation and profitability?\n5. How do Tesla’s capital expenditure plans align with its strategic shift towards software and autonomous services in the medium term?" follow_up_questions=['How sustainable is Tesla’s revenue growth in the face of increasing competition in both electric vehicles and autonomous software?', "What are the critical milestones and regulatory hurdles impacting the scaling and monetization of Tesla's Full Self-Driving subscriptions?", 'How will Tesla manage cost pressures and supply chain challenges to stabilize or improve automotive gross margins?', 'What impact could macroeconomic factors such as interest rate changes and inflation have on Tesla’s valuation and profitability?', 'How do Tesla’s capital expenditure plans align with its strategic shift towards software and autonomous services in the medium term?']

In [13]:
# Create a session to store the handoff history
session = SQLiteSession("research_agent_handoff")

In [14]:
# Define the input schema for transferring from Planner to Writer
class PlannerToWriterInput(BaseModel):
    original_query: str  # The original user query string
    search_plan: SearchPlan  # The search plan generated by the Planner


# Callback function that runs when Planner hands off to Writer
def on_planner_to_writer(ctx: RunContextWrapper[None], input_data: PlannerToWriterInput):
    print_markdown("➡️ Transfer: Planner → Writer")  # Print a message indicating the handoff took place


# Define the actual handoff setup from Planner to Writer
handoff_to_writer = handoff(
    agent = writer_agent,                  # The target agent (Writer) that will receive the handoff
    input_type = PlannerToWriterInput,     # The type of input data expected by the Writer
    on_handoff = on_planner_to_writer,     # The callback triggered during handoff (prints the transfer message)
    tool_name_override = "transfer_to_writer",       # Custom tool name for the handoff
    tool_description_override = "Transfer to Writer with original query and search plan")  # Tool description


# Clone the Planner agent and add handoff instructions + the handoff tool
# This ensures the Planner knows how to call the Writer once the search plan is ready.
planner_with_handoff = planner_agent.clone(
    instructions=f"""{RECOMMENDED_PROMPT_PREFIX}\n\n"""
    + planner_agent.instructions
    + "\n\nWhen you have produced the SearchPlan, call the handoff tool `handoff_to_writer` with this JSON input: {{ original_query: <the user query>, search_plan: <the SearchPlan JSON> }}.\n",
    handoffs=[handoff_to_writer])  # Add the handoff to Writer


In [15]:
"""
This is the main function that runs the handoff chain end-to-end with a sample query.
"""
async def run_handoffs_demo(user_query: str):
    print_markdown(f"## 🕵️‍♀️ User Query\n{user_query}")  # Display the user query

    # Start the chain at the Planner; it will handoff to Writer
    run_res = await Runner.run(planner_with_handoff, user_query, session = session)
    print_markdown("---")  # Separator for output

    # Check if the final output is from the Verifier and display the result
    report = run_res.final_output

    # Display (This part is correct)
    print_markdown("---")
    print_markdown(f"### 🔎 Executive Summary\n{report.short_summary}")
    print_markdown("\n\n-----------------\n\n")
    print_markdown(f"### 📄 Full Report\n{report.markdown_report}")
    print_markdown("\n\n-----------------\n\n")
    return run_res


In [16]:
# Example usage: run the handoff chain end-to-end with a sample query
handoff_result = await run_handoffs_demo("Can you let me know about the Apple stock")


## 🕵️‍♀️ User Query
Can you let me know about the Apple stock

➡️ Transfer: Planner → Writer

---

---

### 🔎 Executive Summary
Apple's Q1 2026 financial results demonstrate robust revenue growth driven by strong iPhone sales and expanding services, positioning the company well despite mixed macroeconomic conditions and supply chain challenges. Analysts remain cautiously optimistic, projecting significant upside potential with price targets ranging from $256 to $500, supported by innovation and strategic initiatives such as AI integration and U.S. manufacturing investments.



-----------------



### 📄 Full Report
# Comprehensive Investment Report: Apple Inc. (AAPL) Stock – April 2026

## Executive Summary
Apple Inc. closed Q1 2026 with record-breaking quarterly revenue growth of 16%, reaching $143.8 billion, primarily fueled by strong iPhone sales and growth in its services segment. Investor sentiment remains cautiously optimistic amid macroeconomic risks and supply chain pressures, with analyst price targets ranging broadly from $256 to $500, reflecting both confidence in Apple's innovation trajectory and concerns about cost margins and global market volatility.

## Financial Performance Overview
Apple's Q1 2026 performance underscores continued strength in its core hardware business, notably the iPhone 17, which saw a significant surge in global sales in February 2026. This surge contributed to revenue growth estimated between 13% and 16% for Q2 2026, alongside anticipated earnings per share of $1.88 for the March quarter — an approximate 14% year-over-year rise.

Key financial highlights from Q1 2026 include:
- Record quarterly revenue of $143.8 billion, a 16% increase YoY
- Strong acceleration in the services segment complementing hardware sales
- Diversification enabling a robust revenue mix despite some external headwinds

These results set a high comparative benchmark, reflecting Apple's effective product cycles and expanding service offerings.

## Market Sentiment and Analyst Perspectives
Investor attitudes toward Apple stock in April 2026 display a spectrum of cautious optimism. Positive market catalysts include:
- Planned integration of Siri with third-party AI assistants under iOS 27, enhancing user experience and ecosystem stickiness
- A strategic $400 million investment in U.S. manufacturing, signaling supply chain localization and potential cost-control benefits
- Retention bonuses aimed at maintaining key talent against competitor poaching

Wedbush analysts maintain an "Outperform" rating with an ambitious $350 price target based on AI-related growth potential showcased at the recent WWDC event. Other analysts suggest a range of price targets up to $500 in the longer term, contingent on sustained earnings growth and favorable market conditions.

## Risks and Challenges
Despite the upbeat outlook, Apple faces notable risks:
- Macroeconomic headwinds, including rising Brent crude prices (above $112) and rising U.S. Treasury yields (4.46%), contribute to stagflation concerns that could affect consumer spending and market valuations.
- Supply chain margin pressures due to new supplier dynamics, particularly regarding storage components where Apple renegotiated contracts moving away from Kioxia to Samsung at potentially higher costs. This may impact product profitability and returns.
- A roughly 10% year-to-date stock price decline has tempered investor enthusiasm despite strong fundamental performance.

Currently, Apple's price-to-earnings (P/E) ratio stands at 32.2, implying that the market expects continued growth but is factoring in risks associated with economic and operational challenges.

## Strategic Insights
Apple's ability to innovate and adapt remains a critical driver of its market valuation:
- The integration of AI into its ecosystem via Siri and other platform updates may create competitive differentiation.
- The milestone 50-year anniversary this year serves as a reminder of Apple's resilience and the strength of its brand and innovation pipeline.
- Investments in domestic manufacturing could mitigate geopolitical and supply chain risks over time.

## Conclusion
Apple's strong financial performance and sustained innovation underpin its favorable investor outlook, albeit moderated by macroeconomic pressures and supply chain risks. The company's diversified revenue streams and strategic initiatives position it well for medium to long-term growth, but investors should weigh these positives against evolving global economic conditions.

## Follow-Up Research Questions
1. How sustainable are Apple's margin improvements given rising supplier costs and competitive pricing pressures?
2. What is the potential impact of AI integration in Apple's ecosystem on user engagement and revenue diversification?
3. How might macroeconomic factors like rising energy prices and interest rates affect Apple's consumer demand and stock valuation in the near term?
4. What operational benefits and risks does the $400 million U.S. manufacturing investment present?
5. How does Apple's competitive positioning in the smartphone and services markets compare to rivals given recent advancements and product launches?



-----------------



In [17]:
handoff_result = await run_handoffs_demo(
    "Do a deep dive on the latest news and developments in the AAPL stock. Also I need the fundamentals analysis of the company."
)

## 🕵️‍♀️ User Query
Do a deep dive on the latest news and developments in the AAPL stock. Also I need the fundamentals analysis of the company.

➡️ Transfer: Planner → Writer

---

---

### 🔎 Executive Summary
Apple Inc. showcases strong recent financial performance with record quarterly revenue growth and significant earnings increases, supported by robust iPhone sales and expanding services. Despite some volatility and stock price fluctuations linked to delayed product launches and macroeconomic concerns, analysts maintain a cautiously optimistic outlook on Apple stock with price targets generally ranging between $256 and $350. The company's fundamentals reflect solid profitability, healthy growth, and efficient cost management, positioning Apple well for continued success amid competitive and economic challenges.



-----------------



### 📄 Full Report
# Apple Inc. (AAPL) Stock Deep Dive and Fundamentals Analysis – April 2026

## Executive Summary
Apple Inc. delivered a remarkable Q1 2026 earnings report with revenue reaching $143.8 billion, marking a 16% year-over-year rise, and net income of $42.1 billion, reflecting a 19% increase. These financial results were boosted by strong growth in iPhone sales (23% increase) and a 14% rise in its high-margin services segment. Despite some stock volatility driven by product launch delays like the foldable iPhone postponement and tariff-related supply chain risks, market and analyst sentiment remain cautiously optimistic about Apple’s growth prospects, particularly considering AI integrations and upcoming product refresh cycles.

## Recent News and Stock Developments
- **Stock Price and Market Sentiment:** Apple's stock price has experienced fluctuations, with a slight dip of about 4% related to delays in the foldable iPhone launch, now pushed to December 2026. Predictions show price movement in the $256 to $264 range, with some forecasts suggesting an 11% rise to around $287.83 per share later in the year.
- **Analyst Ratings:** Evercore and other leading analysts maintain Apple as a top pick for 2026, citing anticipated enhancements to Siri and the upcoming iPhone cycle refreshes as catalysts for growth.
- **Macroeconomic and Geopolitical Risks:** Concerns around tariffs, particularly with supply chain exposure in China, and broader macroeconomic conditions such as inflation and interest rate pressures, contribute to cautious sentiment.
- **Innovation and Product Strategy:** AI upgrades integrated into Siri and other Apple services are viewed as key long-term growth drivers. Challenges include the underperformance of the Vision Pro headset and halted Apple Car project development, balancing optimism with pragmatic caution.

## Financial Fundamentals Analysis
- **Revenue and Earnings Growth:** Apple’s latest quarter surpassed expectations, with revenue at $143.8 billion (16% YoY growth) and net income of $42.1 billion (19% YoY growth). Diluted earnings per share stood at $2.84, up 19% from last year.
- **Segment Performance:** iPhone revenue surged by 23%, while the Services segment exhibited a 14% increase, both contributing significantly to top-line growth.
- **Profitability:** Apple's profit margin remains robust at approximately 27.04%, outperforming most peers within the technology sector.
- **Long-Term Growth Metrics:** Over five years, Apple has achieved an 8.7% revenue CAGR and 14.3% net income CAGR, supported by efficient cost management and premium product offerings.
- **Capital Management:** Strong capital returns include dividends and share buybacks, aligning with shareholder interests.
- **Diversified Revenue Streams:** Beyond hardware, Apple's ecosystem expansion through services like Apple Music, iCloud, and Pay alongside wearables like AirPods and Apple Watch bolsters margin stability and user retention.

## Risks and Considerations
- **Product Launch Delays:** The foldable iPhone delay may impact short-term market excitement and revenue timing.
- **Supply Chain and Tariff Exposure:** Apple's dependence on components from China and associated tariffs create cost and margin pressures.
- **Market Volatility:** Inflation, rising energy prices, and interest rate fluctuations add uncertainty to consumer spending and valuation multiples.

## Conclusion
Apple's solid financial and operational fundamentals combined with its innovation pipeline and strategic market positioning present a compelling investment case. While stock volatility and external risks suggest caution, the company's resilient revenue streams, expanding ecosystem, and anticipated technological advancements provide strong growth catalysts.

## Follow-Up Research Questions
1. How will the delayed foldable iPhone launch affect Apple's revenue and market positioning in 2026?
2. What specific impacts do tariff-related supply chain risks have on Apple's cost structures and margin forecasts?
3. How significant will AI and Siri enhancements be in diversifying Apple's revenue beyond traditional hardware sales?
4. How do macroeconomic factors, including interest rate trends and inflation, influence Apple’s consumer demand outlook?
5. What are the expectations for Apple’s capital return programs given recent earnings growth and cash flow generation?



-----------------

